## Experiments of the project

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [2]:
## Load the dataset
data = pd.read_csv("../Datasets/Churn_Modelling.csv")

print(data.shape)
data.head()

(10000, 14)


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Feature Engineering

In [3]:
## Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

data.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')

In [4]:
## Encode categorical variables
le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])

ohe = OneHotEncoder(sparse_output=False)
geo_encoded_value = ohe.fit_transform(data[['Geography']])

In [5]:
data['Gender'].head()

0    0
1    0
2    0
3    0
4    0
Name: Gender, dtype: int64

In [6]:
## Encoded values
geo_encoded_value

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [7]:
## Get feature names after encoding
ohe.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
## Convert to dataframe
geo_encoded_df = pd.DataFrame(data=geo_encoded_value, columns=ohe.get_feature_names_out())

geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [9]:
## Merge this to the original data
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [10]:
## Save the encoders
with open('../artifacts/gender_label_encoder.pkl', 'wb') as file:
    pickle.dump(le, file)

with open('../artifacts/geo_onehot_encoder.pkl', 'wb') as file:
    pickle.dump(ohe, file)

In [11]:
## Divide the dataset into features and target
X = data.drop('Exited', axis=1)
y = data['Exited']

## Split the data into training and testing set
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

## Scale the feature
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

In [12]:
X_test

array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.81311987, -1.09499335,  0.77030065, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.41876609,  0.91324755, -0.94100321, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.24540869,  0.91324755,  0.00972116, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(2000, 12))

In [13]:
## Save scalar as pickle file
with open('../artifacts/scalar.pkl', 'wb') as file:
    pickle.dump(scalar, file)

### ANN Implementation

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [15]:
## Build the ANN Model
model = Sequential([
    Dense(units=64, activation='relu', input_shape=(X_train.shape[1],)), # HL1 connected with input layer
    Dense(32, activation='relu'),   # HL2
    Dense(1, activation='sigmoid')  # output layer

])

## Model summary
model.summary()

e:\AI\03 - Deep Learning\Deep-Learning-Projects\ANN\Churn-Modelling\churn_venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
## Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) 

## If you using optimizer, loss like this way you don't have any control on it's parameters

## That's why you should instantiate these

In [17]:
## Define optimizer, lossess, etc
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
# loss = tf.keras.losses.BinaryCrossentropy()
# model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy']) 

In [18]:
## Set up the Tensorboard
# To visualize and track all log files of training
# You can also do the samething with matplotlib but tensorboard is the good one

log_dir = "../logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callbacks = TensorBoard(log_dir=log_dir, histogram_freq=1)    # initialize tensorboard

In [19]:
## Setup Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [20]:
## Train the model
history = model.fit(
    X_train, y_train, 
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorflow_callbacks, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8052 - loss: 0.4569 - val_accuracy: 0.8275 - val_loss: 0.3969
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8363 - loss: 0.3932 - val_accuracy: 0.8505 - val_loss: 0.3624
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8505 - loss: 0.3620 - val_accuracy: 0.8620 - val_loss: 0.3467
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8555 - loss: 0.3483 - val_accuracy: 0.8555 - val_loss: 0.3468
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8569 - loss: 0.3423 - val_accuracy: 0.8590 - val_loss: 0.3424
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8609 - loss: 0.3387 - val_accuracy: 0.8570 - val_loss: 0.3462
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8612 - loss: 0.3348 - val_accuracy: 0.8570 - val_loss: 0.3403
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8621 - loss: 0.3336 - val_accu

In [21]:
## Save the model file
model.save("../artifacts/model.h5")

In [26]:
## Load the Tensorboard Extension
%load_ext tensorboard

## It is a Jupyter/IPython magic command that loads the TensorBoard notebook extension into the current notebook session.

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
## Now opens the TensorBoard dashboard for the log folder
%tensorboard --logdir ../logs/

## See the output of this cell in the browser "http://localhost:6006"

## Stop the process
# !tasklist | findstr "tensorboard"
# !taskkill /F /PID <PID>


In [ ]:
## See the process id
!tasklist | findstr "tensorboard"


tensorboard.exe              14504 Console                    1      6,776 K


In [ ]:
## Stop the process with <PID>
# <PID> is the first number of above output e.g., 14504
!taskkill /F /PID 14504

SUCCESS: The process with PID 14504 has been terminated.


### Prediction

In [32]:
## We will do it in another file
